# LangGraph trace-native optimization demo

Compact, deterministic demo for `backend="trace"` without OTEL ingestion.


In [ ]:
from langgraph.graph import StateGraph, START, END
from opto.trace import node
from opto.trace.io import instrument_graph, optimize_graph

def _raw(value):
    return getattr(value, 'data', value)


In [ ]:
planner_prompt = node('Create a plan for: {query}', trainable=True, name='planner_prompt')
synth_prompt = node('Answer: {query}\nPlan: {plan}', trainable=True, name='synth_prompt')

def planner_node(state):
    query = _raw(state['query'])
    return {'query': str(query), 'plan': planner_prompt.data.replace('{query}', str(query))}

def synth_node(state):
    query = _raw(state['query'])
    plan = _raw(state['plan'])
    answer = synth_prompt.data.replace('{query}', str(query)).replace('{plan}', str(plan))
    return {'final_answer': node(answer, name='final_answer_node')}

def build_graph():
    g = StateGraph(dict)
    g.add_node('planner', planner_node)
    g.add_node('synth', synth_node)
    g.add_edge(START, 'planner')
    g.add_edge('planner', 'synth')
    g.add_edge('synth', END)
    return g


In [ ]:
ig = instrument_graph(
    backend='trace',
    graph_factory=build_graph,
    scope=globals(),
    graph_agents_functions=['planner_node', 'synth_node'],
    graph_prompts_list=[planner_prompt, synth_prompt],
    output_key='final_answer',
)
result = ig.invoke({'query': 'What is CRISPR?'})
assert 'CRISPR' in result['final_answer'].data
result['final_answer'].data


In [ ]:
class NotebookOptimizer:
    def __init__(self, prompt):
        self.prompt = prompt
        self.calls = 0

    def zero_feedback(self):
        return None

    def backward(self, *_args, **_kwargs):
        return None

    def step(self):
        self.calls += 1
        if self.calls == 1:
            self.prompt._data = 'CRISPR optimized :: {query} :: {plan}'
            return {'synth_prompt': self.prompt._data}
        return {}

def eval_fn(payload):
    answer = str(payload['answer'])
    return {
        'score': 1.0 if 'CRISPR optimized' in answer else 0.0,
        'feedback': 'Prefer mentioning CRISPR optimized explicitly.',
    }

opt = optimize_graph(
    ig,
    queries=['What is gene editing?'],
    iterations=2,
    optimizer=NotebookOptimizer(synth_prompt),
    eval_fn=eval_fn,
)
assert opt.best_iteration == 2
assert opt.best_score == 1.0
assert opt.best_updates['synth_prompt'].startswith('CRISPR optimized')
opt.best_score, opt.best_updates
